# 16 · Excel mit openpyxl

**Ausblick:** Dieses Notebook ist eine Vorführung ohne Übungen.

**Lernziele**

- Du kennst den Aufbau einer Excel-Datei: Workbook, Worksheet, Cell.
- Du liest einzelne Zellen und Bereiche gezielt aus.
- Du findest eine Tabelle im Blatt, auch wenn Titelzeilen darüber stehen.
- Du schreibst einen Bericht mit Formatierung und Formeln.
- Du weißt, warum eine Formel beim Zurücklesen leer sein kann.

## pandas oder openpyxl?

In Notebook 15 hat pandas eine ganze Excel-Tabelle auf einmal eingelesen. Das ist ideal für Auswertungen. Manchmal brauchst du aber das Excel-Blatt selbst: eine bestimmte Zelle, einen Bereich, Schrift und Farben, Formeln, verbundene Zellen. Dafür gibt es **openpyxl**; pandas benutzt es übrigens selbst, um `.xlsx`-Dateien zu lesen.

| Aufgabe | besser mit |
| --- | --- |
| Tabelle auswerten: filtern, gruppieren, rechnen | pandas |
| bestimmte Zellen lesen, Tabelle im Blatt suchen | openpyxl |
| Bericht mit Formatierung, Formeln, Spaltenbreiten | openpyxl |
| vorhandene Datei ändern, ohne ihr Layout zu verlieren | openpyxl |

openpyxl liest und schreibt nur das moderne Format `.xlsx`, keine alten `.xls`-Dateien. Excel selbst braucht es nicht: Es funktioniert auch auf Rechnern ohne Office.

## Aufbau einer Excel-Datei

openpyxl bildet eine Excel-Datei mit drei Klassen nach:

| Klasse | steht für | holst du mit |
| --- | --- | --- |
| `Workbook` | die ganze Datei | `load_workbook("datei.xlsx")` |
| `Worksheet` | ein Tabellenblatt | `wb["Blattname"]` |
| `Cell` | eine Zelle mit Wert und Format | `ws["A5"]` |

![Ausschnitt aus dem Umsatzbericht: Spalten A bis D, Zeilen 1 bis 6. A1 bis D1 sind zu einer Titelzelle verbunden, in Zeile 4 stehen die Spaltennamen. Die Zelle A5 (Köln) ist grün umrandet: ws["A5"]. Der Bereich A4 bis B6 ist orange umrandet: ws["A4:B6"], drei Zeilen mit je zwei Zellen. Unten die Blätter Umsatz 2026 und Ziele.](../assets/16_excel_aufbau.png)

Wir arbeiten wieder mit dem Umsatzbericht aus Notebook 15:

In [ ]:
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill

wb = load_workbook("../daten/umsatz_filialen.xlsx")
print(wb.sheetnames)

ws = wb["Umsatz 2026"]
print(ws.dimensions)  # genutzter Bereich
print(ws.max_row, "Zeilen,", ws.max_column, "Spalten")

## Einzelne Zellen

Eine Zelle holst du über ihre Adresse wie aus einem Dictionary. Das Ergebnis ist ein `Cell`-Objekt; den Inhalt liefert `.value`. Alternativ zählst du mit `cell(row=..., column=...)`, praktisch in Schleifen. Beide Wege zählen ab 1, nicht ab 0:

In [ ]:
print(ws["A1"].value)
print(ws["A2"].value)
print(ws["A5"].value, ws["C5"].value)

cell = ws.cell(row=5, column=3)
print(cell.coordinate, cell.value)

Der Titel in Zeile 1 ist über die ganze Breite verbunden. Den Wert trägt nur die erste Zelle des verbundenen Bereichs; die übrigen sind leer:

In [ ]:
print(ws.merged_cells.ranges)
print(ws["B1"].value)

## Bereiche

`ws["A4:D6"]` liefert die Zellen eines Rechtecks als Tupel von Zeilen; jede Zeile ist wieder ein Tupel von Zellen. Also zwei Schleifen ineinander wie in Notebook 04, oder eine Schleife mit List Comprehension:

In [ ]:
for row in ws["A4:D6"]:
    print([cell.value for cell in row])

## Die Tabelle im Blatt finden

In Notebook 15 haben wir pandas mit `header=3` gesagt, wo die Spaltennamen stehen. Fügt die Fachabteilung nächsten Monat eine weitere Titelzeile ein, stimmt die 3 nicht mehr. Robuster: Das Programm sucht die Zeile selbst. `ws["A"]` liefert alle Zellen der Spalte A; die erste mit `Filiale` ist die Kopfzeile:

In [ ]:
header_row = None
for cell in ws["A"]:
    if cell.value == "Filiale":
        header_row = cell.row
        break

header = [cell.value for cell in ws[header_row]]
print(header_row, header)

Ab der Zeile darunter lesen wir Filiale und Summe in ein Dictionary. `iter_rows(values_only=True)` liefert jede Zeile direkt als Tupel von Werten. Die Tabelle endet an der ersten Zeile ohne Region, also bei der Leerzeile vor `Gesamt`:

In [ ]:
totals = {}
for name, region, *months, total in ws.iter_rows(min_row=header_row + 1, values_only=True):
    if region is None:
        break
    totals[name] = total

totals

## Einen Bericht schreiben

`Workbook()` legt eine neue, leere Datei mit einem Blatt an. `append()` hängt eine Zeile unten an, wie bei einer Liste. Schrift und Hintergrund setzt du je Zelle mit `Font` und `PatternFill`, das Zahlenformat mit `number_format` (dieselben Codes wie in Excel unter *Zellen formatieren*). Die Gesamtsumme schreiben wir als Formel, damit sie in Excel mitrechnet, wenn jemand eine Zahl ändert:

In [ ]:
report = Workbook()
sheet = report.active
sheet.title = "Bericht"

sheet["A1"] = "Umsatz Januar bis September 2026"
sheet["A1"].font = Font(bold=True, size=14)
sheet.append([])
sheet.append(["Filiale", "Umsatz"])
for cell in sheet[3]:
    cell.font = Font(bold=True, color="FFFFFF")
    cell.fill = PatternFill("solid", fgColor="4472C4")

for name, total in totals.items():
    sheet.append([name, total])
last_row = sheet.max_row
sheet.append(["Gesamt", f"=SUM(B4:B{last_row})"])
sheet[f"A{last_row + 1}"].font = Font(bold=True)

for (cell,) in sheet.iter_rows(min_row=4, min_col=2, max_col=2):
    cell.number_format = "#,##0 €"
sheet.column_dimensions["A"].width = 14
sheet.column_dimensions["B"].width = 16
sheet.freeze_panes = "A4"  # Kopf bleibt beim Scrollen stehen

report_path = "../daten/ausgabe/bericht_filialen.xlsx"
report.save(report_path)
print("gespeichert:", report_path, "mit Formel in Zeile", last_row + 1)

Öffne die Datei `daten/ausgabe/bericht_filialen.xlsx` in Excel: Die Gesamtsumme steht da, der Kopf ist blau, die Beträge haben Tausenderpunkte und Euro-Zeichen.

## Formel oder Wert?

Liest du dieselbe Datei mit Python zurück, erlebst du eine Überraschung:

In [ ]:
total_cell = f"B{last_row + 1}"
formulas = load_workbook(report_path)
values = load_workbook(report_path, data_only=True)
print(formulas["Bericht"][total_cell].value)
print(values["Bericht"][total_cell].value)

![Eine Formelzelle speichert die Formel und ihr letztes Ergebnis. Mit openpyxl geschrieben ist das Ergebnis leer: load_workbook liefert die Formel als Text, mit data_only=True None. Nachdem Excel die Datei geöffnet und gespeichert hat, liefert data_only=True das Ergebnis 7705951.](../assets/16_formel_wert.png)

openpyxl ist kein Excel: Es rechnet keine Formeln aus. Ohne Angabe bekommst du die Formel als Text, mit `data_only=True` das zuletzt gespeicherte Ergebnis. Das gibt es aber erst, wenn Excel die Datei einmal geöffnet und gespeichert hat. Dasselbe gilt für pandas, das intern openpyxl benutzt. Brauchst du die Zahl gleich in Python, rechne sie in Python aus:

In [ ]:
sum(totals.values())

## Eine vorhandene Datei ändern

openpyxl kann auch eine fertige Datei laden, einzelne Zellen ändern und wieder speichern; Titel, verbundene Zellen und Formate bleiben erhalten. Typische Aufgabe: Alle Monatsumsätze unter 60.000 € sollen rot hinterlegt werden. Die Monate stehen in den Spalten C bis K, die Filialen in den acht Zeilen unter der Kopfzeile. Gespeichert wird unter neuem Namen, damit das Original unangetastet bleibt:

In [ ]:
red = PatternFill("solid", fgColor="F8CBAD")
marked = []
for row in ws.iter_rows(min_row=header_row + 1, max_row=header_row + 8, min_col=3, max_col=11):
    for cell in row:
        if cell.value < 60_000:
            cell.fill = red
            marked.append(cell.coordinate)

wb.save("../daten/ausgabe/umsatz_markiert.xlsx")
print(len(marked), "Zellen markiert:", marked)

Öffne `daten/ausgabe/umsatz_markiert.xlsx`: Der Bericht sieht aus wie vorher, nur die schwachen Monate von Siegen, Wetzlar und Gießen sind rot.

## Zusammenfassung

- pandas für Auswertungen ganzer Tabellen, openpyxl für einzelne Zellen, Formate, Formeln und das Ändern vorhandener Dateien.
- `load_workbook()` öffnet eine Datei (`Workbook`), `wb["Blatt"]` holt ein `Worksheet`, `ws["A5"]` eine `Cell`; den Inhalt liefert `.value`. Zeilen und Spalten zählen ab 1.
- `ws["A4:D6"]` liefert Zeilen von Zellen, `iter_rows(values_only=True)` Zeilen von Werten.
- Statt feste Zeilennummern anzunehmen, sucht das Programm die Kopfzeile selbst.
- Schreiben: `Workbook()`, `append()`, `Font`, `PatternFill`, `number_format`, `column_dimensions`, `freeze_panes`, `save()`.
- openpyxl rechnet keine Formeln aus: `data_only=True` liefert nur Ergebnisse, die Excel gespeichert hat.